# Модуль 07 · Урок 4. Тестирование pandas-кода и контракты данных

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Что должен проверять тест
Автотест — не сравнение исходных строчек кода с эталоном, а проверка **поведения**: значений, порядка строк и колонок, типов, индекса, сохранности входных данных и граничных случаев. Для pandas существуют `pandas.testing.assert_frame_equal` и `assert_series_equal`, которые гораздо точнее обычного сравнения списков. Проверяйте на небольших специально подобранных таблицах, где видны ошибки.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
expected = pd.DataFrame({'qty':[3,5]})
actual = pd.DataFrame({'qty':[3,5]})
assert_frame_equal(actual,expected)

## 2. Контракт функции
Хорошая функция отвечает на конкретный вопрос и имеет понятный контракт: входные колонки, их типы, порядок, что делать с NA, меняется ли оригинал, какие ошибки допустимы. Обработка DataFrame зависит не только от значений, но и от индексов. Если возвращается Series для объединения с исходной таблицей, объясните правило выравнивания.

## 3. Проверка схемы и бизнес-инвариантов
Можно проверять не только равенство, но и свойства: `order_id.is_unique`, `qty.ge(0).all()`, отсутствие пропусков в обязательных полях, допустимые статусы. Эти свойства называют **инвариантами** — то, что обязано выполняться после каждого этапа. При нарушении лучше завершать шаг с явной ошибкой и номером проблемной записи.

In [ ]:
sample = pd.DataFrame({'id':['x','y'],'qty':[2,3]})
assert sample.id.is_unique
assert sample.qty.ge(0).all()

## 4. Тесты на неизменность входа и границы
Подавайте на вход пустые таблицы, одну строку, повторяющиеся ключи, нули, NA, необычные индексы, другую последовательность колонок. При этом тест не должен быть чрезмерно связан с реализацией: использование `groupby` или `pivot_table` не важно, если одинаковый корректный результат. Для больших проектов дополнительные тесты должны проверять время обработки и RAM в контролируемых условиях, не делая произвольных жёстких лимитов.

**Проверьте себя:** почему тест на одном «идеальном» DataFrame недостаточен? Чем инвариант отличается от проверки одной ожидаемой таблицы? Почему следует тестировать неизменность входа?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m7_16 · validate schema

Получите True, если ВСЕ колонки required присутствуют в таблице и False иначе.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `required` — список обязательных колонок; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `required`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_16: используются переменные df, required

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

required = ['order_id', 'qty']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_16 = expected_for("m7_16", df=df, required=required)


In [ ]:
# m7_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, required
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`bool`. Результат для показанного примера: `True`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_16", result, _expected_m7_16)

### m7_17 · validate unique

Получите bool: в column нет повторяющихся значений, включая NA как возможное повторяющееся значение.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `column` — название столбца (`str`); в этом примере: `'order_id'`.


**Выполнение:** используйте готовые переменные `df`, `column`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_17: используются переменные df, column

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

column = 'order_id'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_17 = expected_for("m7_17", df=df, column=column)


In [ ]:
# m7_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, column
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`bool`. Результат для показанного примера: `True`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_17", result, _expected_m7_17)

### m7_18 · assert clean orders

Получите dict флагов `valid_prices`, `valid_quantities`, `unique_ids` (значения Python bool).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_18: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_18 = expected_for("m7_18", df=df)


In [ ]:
# m7_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'valid_prices': True, 'valid_quantities': True, 'unique_ids': True}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_18", result, _expected_m7_18)

### m7_19 · test counts

Получите dict `added` — число новых ID в after, `removed` — число ID из before, которых нет в after (сравнивать уникальные order_id).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `before` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `after` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `before`, `after`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_19: используются переменные before, after

before = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4'],
    'client_id': [1, 2, 1, 3],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги'],
    'price': [127.0, 247.0, 97.0, 317.0],
    'qty': [2, 1, 4, 1],
    'discount': [0, 10, 5, 0],
    'status': ['paid', 'new', 'paid', 'paid'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок'],
}, index=['z', 'b', 'a', 'd'])

after = pd.DataFrame({
    'order_id': ['demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 3, 2, 4],
    'city': ['Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'книги', 'кофе', 'чай'],
    'price': [97.0, 317.0, 57.0, 147.0],
    'qty': [4, 1, 3, 0],
    'discount': [5, 0, 20, 0],
    'status': ['paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['Срочно', 'подарок', '', None],
}, index=['a', 'd', 'x', 'q'])

print("before — демонстрационные данные:")

display(before)

print("after — демонстрационные данные:")

display(after)


# Снимок правильного результата для автопроверки.
_expected_m7_19 = expected_for("m7_19", before=before, after=after)


In [ ]:
# m7_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: before, after
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'added': 2, 'removed': 2}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_19", result, _expected_m7_19)

### m7_20 · quality score

Получите долю от 0 до 1 строк, где price>=0, qty>=0, status не NA; при пустой таблице получите 1.0.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_20: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m7_20 = expected_for("m7_20", df=df)


In [ ]:
# m7_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`float`. Результат для показанного примера: `1.0`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_20", result, _expected_m7_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()